# Practice #5. "Machine Learning for Time Series Forecasting"

This notebook is dedicated to:
* Feature Engineering for Time Series
* Linear Regression for Time Series Forecasting
* Support Vector Machine (SVM) for Time Series Forecasting
* Model Comparison and Evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')

## 0. Data reading and visualization

Please, specify path to data

In [ ]:
path_to_datafile = "../data/daily-total-female-births.csv"

In [ ]:
# data reading to pandas.DataFrame
df = pd.read_csv(path_to_datafile)

Please, rename time column to `ds` and data column to `y`(you can use `df.rename`) . If use dataset with multiple features select only one and drop NaN values

In [ ]:
# your code here
# df.rename(columns={"...": "ds", "...": "y"}, inplace=True)

Convert date column to datetime format and set as index

In [ ]:
df["ds"] = pd.to_datetime(df["ds"])
df.set_index("ds", inplace=True)

Number of data data points:

In [ ]:
df.shape[0]

Print slice of the timeseries:

In [ ]:
df.head()

Let's plot the data

In [ ]:
plt.figure(figsize=(20, 5))
plt.ylabel("y")
plt.xlabel("ds")
plt.plot(df);

## 1. Feature Engineering for Time Series

Machine Learning algorithms require features (X) to predict targets (y). For time series, we need to transform the sequential data into a supervised learning problem. This involves creating features from:

1. **Lagged values**: Previous observations
2. **Rolling statistics**: Moving averages, standard deviations
3. **Time-based features**: Year, month, day, weekday
4. **Trend and seasonality**: Linear trend, seasonal indicators
5. **Technical indicators**: Rate of change, momentum

### 1.1 Feature Engineering Functions

In [ ]:
def create_time_features(df):
    """
    Create time-based features from datetime index
    """
    features = pd.DataFrame(index=df.index)
    
    # Basic time features
    features['year'] = df.index.year
    features['month'] = df.index.month
    features['day'] = df.index.day
    features['dayofweek'] = df.index.dayofweek
    features['dayofyear'] = df.index.dayofyear
    
    # Cyclical features (sine and cosine transformations)
    features['month_sin'] = np.sin(2 * np.pi * features['month'] / 12)
    features['month_cos'] = np.cos(2 * np.pi * features['month'] / 12)
    features['day_sin'] = np.sin(2 * np.pi * features['day'] / 31)
    features['day_cos'] = np.cos(2 * np.pi * features['day'] / 31)
    
    # Linear trend
    features['trend'] = np.arange(len(df))
    
    return features

def create_lag_features(series, n_lags=12):
    """
    Create lagged features from time series
    """
    features = pd.DataFrame(index=series.index)
    
    for lag in range(1, n_lags + 1):
        features[f'lag_{lag}'] = series.shift(lag)
    
    return features

def create_rolling_features(series, windows=[3, 6, 12]):
    """
    Create rolling window features
    """
    features = pd.DataFrame(index=series.index)
    
    for window in windows:
        features[f'rolling_mean_{window}'] = series.rolling(window=window).mean()
        features[f'rolling_std_{window}'] = series.rolling(window=window).std()
        features[f'rolling_min_{window}'] = series.rolling(window=window).min()
        features[f'rolling_max_{window}'] = series.rolling(window=window).max()
    
    return features

def create_diff_features(series, periods=[1, 12]):
    """
    Create differencing features
    """
    features = pd.DataFrame(index=series.index)
    
    for period in periods:
        features[f'diff_{period}'] = series.diff(periods=period)
        features[f'pct_change_{period}'] = series.pct_change(periods=period)
    
    return features

### 1.2 Create Features Dataset

Please, create a comprehensive feature set for your time series:

In [ ]:
# your code here
# Create all features
# ...

# Combine all features
# ...

# Add target variable
# ...

# Remove rows with NaN values
# ...

# print(f"Feature matrix shape: {features.shape}")

### 1.3 Prepare Training Data

Split features and target, then create train/test sets:

In [ ]:
# your code here
# Separate features and target
# ...

# Time series split (important: maintain temporal order)
# ...

# print(f"Training set: {X_train.shape}")
# print(f"Test set: {X_test.shape}")

## 2. Linear Regression for Time Series

Linear regression models assume a linear relationship between features and target:

$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n + \epsilon$$

For time series, this becomes:
$$y_t = \beta_0 + \beta_1 y_{t-1} + \beta_2 y_{t-2} + ... + \beta_k \text{features}_t + \epsilon_t$$

### 2.1 Basic Linear Regression

In [ ]:
# your code here
# Scale features for better performance
# ...

# Train linear regression model
# ...

# Make predictions
# ...

# Calculate metrics
# ...

# print(f"Linear Regression Results:")
# print(f"RMSE: {rmse_lr:.4f}")
# print(f"MAE: {mae_lr:.4f}")
# print(f"R²: {r2_lr:.4f}")

### 2.2 Regularized Linear Regression

Apply Ridge and Lasso regression to handle potential overfitting:

In [ ]:
# your code here
# Ridge Regression (L2 regularization)
# ...

# Lasso Regression (L1 regularization)
# ...

# Calculate metrics for both models
# ...

# print(f"Ridge Regression RMSE: {rmse_ridge:.4f}")
# print(f"Lasso Regression RMSE: {rmse_lasso:.4f}")

# Feature importance from Lasso (non-zero coefficients)
# ...
# print("\nTop 10 Important Features (Lasso):")
# ...

### 2.3 Polynomial Regression

Use polynomial features to capture non-linear relationships:

In [ ]:
# your code here
# Create polynomial pipeline
# ...

# Train polynomial model
# ...

# ...
# print(f"Polynomial Regression RMSE: {rmse_poly:.4f}")

## 3. Support Vector Machine (SVM) for Time Series

Support Vector Regression (SVR) can capture non-linear patterns using kernel functions. The key idea is to map the input space to a higher-dimensional space where linear regression can be applied.

**Key SVR Parameters:**
- **C**: Regularization parameter (controls overfitting)
- **epsilon**: Tolerance for error (size of epsilon-tube)
- **kernel**: Type of kernel function (linear, polynomial, RBF, sigmoid)
- **gamma**: Kernel coefficient for RBF, polynomial, and sigmoid

### 3.1 Linear SVR

In [ ]:
# your code here
# Linear SVR
# ...

# rmse_linear_svr = ...
# print(f"Linear SVR RMSE: {rmse_linear_svr:.4f}")

### 3.2 RBF SVR with Hyperparameter Tuning

In [ ]:
# your code here
# RBF SVR with Grid Search
# param_grid = ...

# # Use TimeSeriesSplit for cross-validation
# ...



# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best CV score: {-grid_search.best_score_:.4f}")

# Best RBF SVR model
# ...

# rmse_rbf_svr = ...
# print(f"RBF SVR RMSE: {rmse_rbf_svr:.4f}")

### 3.3 Polynomial SVR

In [ ]:
# your code here
# Polynomial SVR
# ...

# rmse_poly_svr = ...
# print(f"Polynomial SVR RMSE: {rmse_poly_svr:.4f}")

### 4.4 Questions for Analysis

**Questions to consider:**

1. Which model performed best and why?
2. How do linear models compare to SVR models?
3. What features were most important for prediction?
4. Are there patterns in the residuals that suggest model improvements?
5. How does regularization affect model performance?
6. Which kernel function worked best for SVR and why?
7. How do these ML approaches compare to classical time series methods (ARIMA, exponential smoothing)?

**Potential Improvements:**
- Try different feature engineering approaches
- Experiment with ensemble methods
- Consider seasonal decomposition before modeling
- Use cross-validation specifically designed for time series
- Implement walk-forward validation for more realistic performance estimation